In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import hnswlib as hl
import hnsw_benchmark2 as bm
import matplotlib.pyplot as plt
from datasets import load_dataset
import os

d:\Dokumentumok\MSc\F3_TudLab\EX9BLV_HNSW_Preordering\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
# Adathalmazok beolvasása
corpus = load_dataset("mteb/arguana", "corpus")
corpus_texts = corpus.data["corpus"]["text"]

queries = load_dataset("mteb/arguana", "queries")
query_texts = queries.data["queries"]["text"]

In [ ]:
print(corpus_texts[0])
# print(query_texts[0])

In [ ]:
# Embedding végrehajtása
model = SentenceTransformer("intfloat/e5-small-v2")

corpus_embeddings = model.encode(
    [f"passage: {text}" for text in corpus_texts],
    normalize_embeddings=True,
    batch_size=64,
    show_progress_bar=True,
)

query_embeddings = model.encode(
    [f"query: {text}" for text in query_texts],
    normalize_embeddings=True,
    batch_size=64,
    show_progress_bar=True,
)


corpus_embeddings = np.asarray(corpus_embeddings, dtype=np.float32)
query_embeddings = np.asarray(query_embeddings, dtype=np.float32)

In [ ]:
print(corpus_embeddings.shape)
print(query_embeddings.shape)

In [ ]:
df_corpus = pd.DataFrame(columns=["vectors"], data=[[v] for v in corpus_embeddings])
df_query = pd.DataFrame(columns=["vectors"], data=[[v] for v in query_embeddings])

dataset_name = "arguana"
corpus_filename = dataset_name + "_corpus"
corpus_output_path = os.path.join("..\\data\\", corpus_filename + ".json")
df_corpus.to_json(corpus_output_path, orient="records", lines=True)

query_filename = dataset_name + "_query"
query_output_path = os.path.join("..\\data\\", query_filename + ".json")
df_query.to_json(query_output_path, orient="records", lines=True)

In [ ]:
dataset_name = "arguana"
corpus_filename = dataset_name + "_corpus"
corpus_input_path = os.path.join("..\\data\\", corpus_filename + ".json")
df_corpus = pd.read_json(corpus_input_path, orient="records", lines=True)

query_filename = dataset_name + "_query"
query_input_path = os.path.join("..\\data\\", query_filename + ".json")
df_query = pd.read_json(query_input_path, orient="records", lines=True)

In [ ]:
# KNN számítás minden query vectorra
k = 10
scores = query_embeddings @ corpus_embeddings.T

topk = np.argpartition(
    -scores,
    kth=k - 1,
    axis=1,
)[:, :k]

# Optional: sort those k results by actual score
rows = np.arange(len(query_embeddings))[:, None]
order = np.argsort(-scores[rows, topk], axis=1)

knn_matrix = np.take_along_axis(topk, order, axis=1)

print(knn_matrix.shape)
# for i in knn_matrix[0]:
#     print(f"{i:4d} - dist: {hnswbm.KNN.euclidian_distance(query_embeddings[0], corpus_embeddings[i]):.4f}")

In [ ]:
# HNSW konstruálás
hnsw = hl.Index(space             = 'ip',
                dim               = corpus_embeddings.shape[1])
hnsw.init_index(max_elements      = len(corpus_embeddings),
                M                 = 16,
                ef_construction   = 128)
hnsw.add_items(data               = corpus_embeddings)

In [ ]:
hnsw.set_ef(10)

hnsw_indicies, hnsw_distances = hnsw.knn_query(query_embeddings, k=10)

In [ ]:
# Recall számítás
recalls = []

for R_A, R_E in zip(hnsw_indicies, knn_matrix):
    recalls.append(
        len(set(R_A) & set(R_E)) / len(R_E)
    )

print(np.mean(recalls))

In [ ]:
# HNSW benchmark
bm_random = bm.HNSW_benchmark(data=corpus_embeddings)
recalls_random = bm_random.run(query_embeddings)

print(f"Minimum recall: {np.min(recalls_random):.4f}")
print(f"Average recall: {np.mean(recalls_random):.4f}")
print(f"Maximum recall: {np.max(recalls_random):.4f}")